# Regenerate `df_l1_*` and `df_l2_*` for one lake

This notebook regenerates the lake-level CSV tables from pre-downloaded Landsat C2L1/C2L2 folders. The reusable logic lives in `src/lswt_cloud_masking/lake_tables.py`; this notebook only sets paths, chooses one lake, runs the pipeline, and previews the outputs.

Expected external-drive layout on macOS:

```text
/Volumes/YOUR_EXTERNAL_DRIVE/Trishna/Landsat_processing/Landsat_C2/
  Geneva_L1/
  Geneva_L2_acolite/
  Geneva_L2_usgs/
```

The same pattern is used for `Aegeri`, `Bianco`, `Greifensee`, `Mendota`, and `Venice`.

In [ ]:
from pathlib import Path
import sys

import pandas as pd


def find_repo_root(start=None) -> Path:
    start = start or Path.cwd()
    for path in [start, *start.parents]:
        if (path / "src" / "lswt_cloud_masking").exists():
            return path
    raise RuntimeError("Run this notebook from inside the LSWT-ML-thin-cloud-masking repo.")


REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT / "src"))

from lswt_cloud_masking.lake_tables import generate_lake_tables, load_generation_config


def repo_path(value) -> Path:
    path = Path(value).expanduser()
    return path if path.is_absolute() else REPO_ROOT / path


REPO_ROOT

## Settings

Change `LANDSAT_C2_ROOT` to the mounted external-drive path on your Mac. Use `LIMIT_SCENES = 1` or `2` for a quick smoke test before processing the full lake.

In [ ]:
CONFIG_PATH = REPO_ROOT / "configs" / "lake_tables.example.json"
config = load_generation_config(CONFIG_PATH)

LANDSAT_C2_ROOT = Path("/Volumes/YOUR_EXTERNAL_DRIVE/Trishna/Landsat_processing/Landsat_C2")
LAKE = "geneva"
LIMIT_SCENES = None

lake = next(
    lake
    for lake in config["lakes"]
    if LAKE.lower() in {lake["output_key"].lower(), lake["lake_key"].lower()}
)

lake

## Run

By default the ML-filtered LST columns mask class `1`, matching the old `df_diff_ml_stats_*` workflow. To mask both thin-cloud and cloud-affected classes, change `mask_cloud_classes` to `(1, 2)`.

In [ ]:
report = generate_lake_tables(
    lake=lake,
    landsat_root=LANDSAT_C2_ROOT,
    lake_geojson=repo_path(config["lake_geojson"]),
    model_dir=repo_path(config["models_dir"]),
    output_l1_dir=repo_path(config["output_l1_dir"]),
    output_l2_dir=repo_path(config["output_l2_dir"]),
    mask_cloud_classes=tuple(config.get("mask_cloud_classes", [1])),
    include_cirrus_as_cloud=bool(config.get("include_cirrus_as_cloud", False)),
    include_lake_metadata_in_l1=bool(config.get("include_lake_metadata_in_l1", False)),
    limit_scenes=LIMIT_SCENES,
    continue_on_error=bool(config.get("continue_on_error", True)),
)

report

## Preview outputs

In [ ]:
df_l1 = pd.read_csv(report["l1_csv"])
df_l2 = pd.read_csv(report["l2_csv"])

print("L1", df_l1.shape, report["l1_csv"])
display(df_l1.head())

print("L2", df_l2.shape, report["l2_csv"])
display(df_l2.head())